In [12]:
import sys
import os

# Add the parent directory to the path so the package is importable
sys.path.append(os.path.abspath(".."))

from llm_data_quality_assistant.enums import Models
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
import numpy as np
import jupyter_helper_functions
import string
import time

load_dotenv()

True

# 2. Load and Explore Flight Data
Load the Flight dataset and perform exploratory data analysis to understand its structure and content.

In [13]:
corrupt_dataset = jupyter_helper_functions.load_dataset(
    "../analysis/repairs/flight/flight-repair.csv"  # the parker repair dataset
)
print(corrupt_dataset.shape)
gold_standard = jupyter_helper_functions.load_dataset(
    "../datasets/parker_datasets/flight/flight_cleaned_gold_first1000_int.csv"
)

# Uncomment these lines for data exploration
# print(corrupt_dataset.head(2))
# print(gold_standard.head(2))
# print(type(gold_standard.get("composed_key").iloc[0]))
# print(type(corrupt_dataset.get("composed_key").iloc[0]))

(24606, 5)


# 3. Clean and Merge Data with LLM
Use the LLM pipeline to clean and merge the corrupted flight dataset and evaluate the results.

In [14]:
from llm_data_quality_assistant.pipeline import Pipeline
from llm_data_quality_assistant.enums import Models
import string
import json
import jupyter_helper_functions

# Use a primary key for merging
primary_key = "composed_key"
model = Models.OpenAIModels.GPT_4_1_MINI
# model = Models.OpenAIModels.GPT_4_1_MINI
rows_of_context = 50

extra = "option_2"
file_name = jupyter_helper_functions.sanitize_filename(f"{model.value}_{rows_of_context}_rows_context_{extra}")   

rpm = 0
additional_prompt = f"""
Here are rows of the dataset to provide context for the cleaning process:
{corrupt_dataset.sample(rows_of_context).to_string(index=False)}
"""

# Merge/clean with LLM
merged_df, time_taken = jupyter_helper_functions.merge_with_llm_timed(
    dataset=corrupt_dataset,
    primary_key=primary_key,
    model=model,
    rpm=rpm,
    additional_prompt=additional_prompt
)
# time_taken = 0

Merging groups with LLM:   0%|          | 0/1000 [00:00<?, ?it/s]

Merging groups with LLM: 100%|██████████| 1000/1000 [24:02<00:00,  1.44s/it] 


In [15]:
# Save merged dataset
# jupyter_helper_functions.save_dataframe_csv(merged_df, f"../analysis/repairs/flight/merged_dataset_{file_name}.csv")

original_corrupted_datasets = jupyter_helper_functions.load_dataset(
    "../datasets/parker_datasets/flight/flight_cleaned_corrupted_first1000_int.csv"
)

# Evaluate results
jupyter_helper_functions.standardize_and_evaluate(
    gold_standard=gold_standard,
    merged_df=merged_df,
    corrupt_dataset=original_corrupted_datasets,
    primary_key=primary_key,
    time_delta=time_taken,
    results_dir=f"../analysis/results/flight/",
    file_name=file_name,
)

{'accuracy': 0.6434101438673494,
 'column_names': ['composed_key',
                  'actual_arrival',
                  'actual_departure',
                  'scheduled_arrival',
                  'scheduled_departure'],
 'f1_score': 0.623725542749933,
 'false_negative': 18374,
 'false_negative_rate': 0.3871226007626994,
 'false_positive': 16723,
 'false_positive_rate': 0.3281529012382018,
 'num_columns': 5,
 'num_rows': 24606,
 'precision': 0.6349646380860909,
 'recall': 0.6128773992373007,
 'time_taken': 1442.8343424797058,
 'true_negative': 34238,
 'true_positive': 29089}
{'column_names': ['composed_key',
                  'actual_arrival',
                  'actual_departure',
                  'scheduled_arrival',
                  'scheduled_departure'],
 'num_columns': 5,
 'num_rows': 24606,
 'stats': [{'accuracy': 0.6556937332357962,
            'column_name': 'actual_arrival',
            'f1_score': 0.6518737672583826,
            'false_negative': 4661,
            'false_n

In [16]:
# Optional: Compare with Parker's repair results
flight_parker_path = "../analysis/repairs/flight/flight-repair.csv"
df_parker = pd.read_csv(flight_parker_path)

# Calculate differences between our repair and Parker's repair
if merged_df is not None:
    column_order = merged_df.columns.tolist()
    df_parker = df_parker[column_order]
    
    diff_mask = (df_parker.values != merged_df.values)
    num_diff = diff_mask.sum()
    total_cells = diff_mask.size
    percent_diff = (num_diff / total_cells) * 100
    
    print(f"Percentage of differing cells between Parker's repair and our LLM repair: {percent_diff:.2f}%")

Percentage of differing cells between Parker's repair and our LLM repair: 16.76%
